In [13]:
import math
import re
import nltk
import pandas as pd
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Required dynamic setups
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

# 1. Specialized Domain Dataset Definition
document_collection = {
    1: {
        "title": "Breakthroughs in Artificial Intelligence",
        "text": "Artificial Intelligence and machine learning are transforming modern healthcare and research analytics.",
    },
    2: {
        "title": "New Innovation in Healthcare Tech",
        "text": "Healthcare systems are adopting machine learning models to improve patient diagnostics and treatment accuracy.",
    },
    3: {
        "title": "Advanced Data Analytics Tools",
        "text": "Data analytics and data science rely heavily on advanced algorithms and statistical processing techniques.",
    },
    4: {
        "title": "Machine Learning Models for Climate",
        "text": "Researchers use machine learning and artificial intelligence to model global climate shifts and weather patterns.",
    },
}

# 2. Text Preprocessing Pipeline
class TextNormalizationEngine:
    def __init__(self):
        self.ignored_words = set(stopwords.words("english"))
        self.word_reducer = PorterStemmer()

    def execute_normalization(self, raw_text):
        lowercased = raw_text.lower()
        cleaned_string = re.sub(r"[^a-z0-9\s]", "", lowercased)
        token_list = cleaned_string.split()

        processed_tokens = [
            self.word_reducer.stem(token)
            for token in token_list
            if token not in self.ignored_words
        ]
        return processed_tokens

# 3. Custom Inverted Index Architecture
class CustomInvertedIndex:
    def __init__(self):
        self.term_postings_map = {}
        self.document_word_counts = {}
        self.text_processor = TextNormalizationEngine()

    def generate_index_matrix(self, target_dataset):
        for doc_id, meta_data in target_dataset.items():
            combined_text = f"{meta_data['title']} {meta_data['text']}"
            extracted_tokens = self.text_processor.execute_normalization(combined_text)

            self.document_word_counts[doc_id] = len(extracted_tokens)

            for token in extracted_tokens:
                if token not in self.term_postings_map:
                    self.term_postings_map[token] = {}

                if doc_id not in self.term_postings_map[token]:
                    self.term_postings_map[token][doc_id] = 1
                else:
                    self.term_postings_map[token][doc_id] += 1

    def fetch_term_records(self, token):
        return self.term_postings_map.get(token, {})

# 4. Search Processor & Vector Space Model
class SpecializedSearchEngine:
    def __init__(self, primary_dataset):
        self.source_data = primary_dataset
        self.index_core = CustomInvertedIndex()
        self.index_core.generate_index_matrix(primary_dataset)
        self.total_document_count = len(primary_dataset)

    def _process_boolean_and(self, query_tokens):
        if not query_tokens:
            return set()

        all_postings = [
            set(self.index_core.fetch_term_records(t).keys()) for t in query_tokens
        ]

        intersection_result = all_postings[0]
        for underlying_set in all_postings[1:]:
            intersection_result = intersection_result.intersection(underlying_set)
        return intersection_result

    def _process_boolean_or(self, query_tokens):
        union_result = set()
        for token in query_tokens:
            postings_data = self.index_core.fetch_term_records(token)
            union_result.update(postings_data.keys())
        return union_result

    def execute_query(self, search_query, operational_mode="OR"):
        tokens_to_search = self.index_core.text_processor.execute_normalization(search_query)

        if not tokens_to_search:
            return []

        if operational_mode.upper() == "AND":
            filtered_documents = self._process_boolean_and(tokens_to_search)
        else:
            filtered_documents = self._process_boolean_or(tokens_to_search)

        matching_scores = {}
        for doc_id in filtered_documents:
            accumulated_score = 0.0
            for token in tokens_to_search:
                current_postings = self.index_core.fetch_term_records(token)
                if doc_id in current_postings:
                    tf_metric = current_postings[doc_id] / self.index_core.document_word_counts[doc_id]
                    document_frequency = len(current_postings)
                    idf_metric = math.log((self.total_document_count / (1 + document_frequency)) + 1)
                    accumulated_score += tf_metric * idf_metric
            matching_scores[doc_id] = accumulated_score

        sorted_evaluations = sorted(matching_scores.items(), key=lambda x: x[1], reverse=True)

        final_payload = []
        for doc_id, calculated_score in sorted_evaluations:
            stored_item = self.source_data[doc_id]
            text_snippet = (
                stored_item["text"][:100] + "..."
                if len(stored_item["text"]) > 100
                else stored_item["text"]
            )
            final_payload.append(
                {
                    "Document ID": doc_id,
                    "Title": stored_item["title"],
                    "Snippet": text_snippet,
                    "Relevance Score": round(calculated_score, 4),
                }
            )

        return final_payload

# 5. Operational Verification Execution
retrieval_system = SpecializedSearchEngine(document_collection)

# Scenario A: OR Logic Evaluation
target_query_or = "machine learning healthcare"
print(f"--- Search Results for (OR Mode): '{target_query_or}' ---")
output_or = retrieval_system.execute_query(target_query_or, operational_mode="OR")
print(pd.DataFrame(output_or).to_string(index=False))

print("\n" + "=" * 80 + "\n")

# Scenario B: AND Logic Evaluation
target_query_and = "artificial intelligence climate"
print(f"--- Search Results for (AND Mode): '{target_query_and}' ---")
output_and = retrieval_system.execute_query(target_query_and, operational_mode="AND")
print(pd.DataFrame(output_and).to_string(index=False))

--- Search Results for (OR Mode): 'machine learning healthcare' ---
 Document ID                                    Title                                                                                                 Snippet  Relevance Score
           2        New Innovation in Healthcare Tech Healthcare systems are adopting machine learning models to improve patient diagnostics and treatment...           0.2054
           1 Breakthroughs in Artificial Intelligence Artificial Intelligence and machine learning are transforming modern healthcare and research analyti...           0.1861
           4      Machine Learning Models for Climate Researchers use machine learning and artificial intelligence to model global climate shifts and weat...           0.1733


--- Search Results for (AND Mode): 'artificial intelligence climate' ---
 Document ID                               Title                                                                                                 Snippet  Rel